In [40]:
import numpy as np

raw_tennis_data = [
    ['Sunny', 'Hot', 'High', 'Weak', 'No'],
    ['Sunny', 'Hot', 'High', 'Strong', 'No'],
    ['Overcast', 'Hot', 'High', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Strong', 'No'],
    ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
    ['Sunny', 'Mild', 'High', 'Weak', 'No'],
    ['Sunny', 'Cool', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'Normal', 'Weak', 'Yes'],
    ['Sunny', 'Mild', 'Normal', 'Strong', 'Yes'],
    ['Overcast', 'Mild', 'High', 'Strong', 'Yes'],
    ['Overcast', 'Hot', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Strong', 'No']
]

mappings = {
    'Sunny': 0, 'Overcast': 1, 'Rain': 2,
    'Hot': 0, 'Mild': 1, 'Cool': 2,
    'High': 0, 'Normal': 1,
    'Weak': 0, 'Strong': 1,
    'No': 0, 'Yes': 1
}

processed_tennis = [[mappings[val] for val in row] for row in raw_tennis_data]
tennis_data = np.array(processed_tennis)

X_tennis = tennis_data[:, :-1]
y_tennis = tennis_data[:, -1]

tennis_feature_names = ['Outlook', 'Temp', 'Humidity', 'Wind']

print(f'X_tennis shape: {X_tennis.shape}')
print(f'y_tennis shape: {y_tennis.shape}')

X_tennis shape: (14, 4)
y_tennis shape: (14,)


In [41]:
import numpy as np

def calculate_entropy(y):
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    entropy = -np.sum(probabilities * np.log2(probabilities))
    return entropy

def calculate_information_gain(y, y_subsets):
    parent_entropy = calculate_entropy(y)

    total_samples = len(y)
    weighted_subset_entropy = 0
    for subset in y_subsets:
        if len(subset) > 0:
            weight = len(subset) / total_samples
            weighted_subset_entropy += weight * calculate_entropy(subset)

    return parent_entropy - weighted_subset_entropy

pure_set = np.array([1, 1, 1, 1])
mixed_set = np.array([0, 1])

entropy_pure = calculate_entropy(pure_set)
entropy_mixed = calculate_entropy(mixed_set)

print(f'Entropy of pure set {pure_set}: {entropy_pure}')
print(f'Entropy of mixed set {mixed_set}: {entropy_mixed}')

parent = np.array([0, 0, 1, 1])
subsets = [np.array([0, 0]), np.array([1, 1])]
ig = calculate_information_gain(parent, subsets)
print(f'Information Gain of perfect split: {ig}')

Entropy of pure set [1 1 1 1]: -0.0
Entropy of mixed set [0 1]: 1.0
Information Gain of perfect split: 1.0


In [42]:
class CategoricalNode:
    def __init__(self, feature_index=None, children=None, value=None):
        self.feature_index = feature_index
        self.children = children if children is not None else {}
        self.value = value

def get_best_feature(X, y, feature_indices):
    best_gain = -1
    best_feature = None

    for feat_idx in feature_indices:
        unique_values = np.unique(X[:, feat_idx])
        subsets = [y[X[:, feat_idx] == val] for val in unique_values]
        gain = calculate_information_gain(y, subsets)

        if gain > best_gain:
            best_gain = gain
            best_feature = feat_idx

    return best_feature

def build_categorical_tree(X, y, feature_indices):
    unique_labels = np.unique(y)
    if len(unique_labels) == 1:
        return CategoricalNode(value=unique_labels[0])

    if len(feature_indices) == 0:
        majority_label = np.bincount(y).argmax()
        return CategoricalNode(value=majority_label)

    best_feat = get_best_feature(X, y, feature_indices)
    if best_feat is None:
        majority_label = np.bincount(y).argmax()
        return CategoricalNode(value=majority_label)

    node = CategoricalNode(feature_index=best_feat, children={})
    remaining_features = [f for f in feature_indices if f != best_feat]

    unique_values = np.unique(X[:, best_feat])
    for val in unique_values:
        val_mask = (X[:, best_feat] == val)
        if np.any(val_mask):
            node.children[val] = build_categorical_tree(X[val_mask], y[val_mask], remaining_features)

    return node

initial_features = list(range(X_tennis.shape[1]))
tennis_tree_root = build_categorical_tree(X_tennis, y_tennis, initial_features)

In [43]:
def predict_single(node, x):
    if node.value is not None:
        return node.value

    feature_val = x[node.feature_index]

    if feature_val in node.children:
        return predict_single(node.children[feature_val], x)
    else:
        return None

def predict_batch(node, X):
    return np.array([predict_single(node, x) for x in X])

y_tennis_pred = predict_batch(tennis_tree_root, X_tennis)

training_accuracy = np.mean(y_tennis_pred == y_tennis)

test_instance = np.array([0, 0, 0, 0])
prediction = predict_single(tennis_tree_root, test_instance)

reverse_label_map = {0: 'No', 1: 'Yes'}
predicted_label = reverse_label_map.get(prediction, 'Unknown')

print(f'Training Accuracy: {training_accuracy * 100:.2f}%')
print(f'Test Case [Sunny, Hot, High, Weak] -> Predicted: {predicted_label}')
print(f'Actual labels:    {y_tennis}')
print(f'Predicted labels: {y_tennis_pred}')

Training Accuracy: 100.00%
Test Case [Sunny, Hot, High, Weak] -> Predicted: No
Actual labels:    [0 0 1 1 1 0 1 0 1 1 1 1 1 0]
Predicted labels: [0 0 1 1 1 0 1 0 1 1 1 1 1 0]
